In [8]:
# fsl_predictor.py
import os
import torch
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader
from PIL import Image

# ====== Config ======
DATA_CHECK = "data/check"
TRAIN_ROOT = "data/train"
MODEL_PATH = "fsl_encoder.pth"
IMAGE_SIZE = 1024
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ====== Encoder Definition (matches your training model) ======
class FSL_Encoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=None)
        base.fc = torch.nn.Identity()
        self.encoder = torch.nn.Sequential(*list(base.children()))
    def forward(self, x):
        for layer in self.encoder:
            x = layer(x)
        return x.mean(dim=[2, 3])  # global average pooling

# ====== Load Encoder ======
encoder = FSL_Encoder()
state_dict = torch.load(MODEL_PATH, map_location=DEVICE)
encoder.load_state_dict(state_dict)
encoder.to(DEVICE)
encoder.eval()

# ====== Transform ======
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ====== Build Class Prototypes ======
train_dataset = datasets.ImageFolder(root=TRAIN_ROOT, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)

features_by_class = {}
with torch.no_grad():
    for imgs, labels in train_loader:
        imgs = imgs.to(DEVICE)
        feats = encoder(imgs)
        for f, label in zip(feats, labels):
            label_name = train_dataset.classes[label]
            if label_name not in features_by_class:
                features_by_class[label_name] = []
            features_by_class[label_name].append(f.cpu())

prototypes = {cls: torch.stack(feats).mean(0) for cls, feats in features_by_class.items()}
class_names = list(prototypes.keys())
print(f"✅ Loaded prototypes: {class_names}")

# ====== Prediction Function ======
def predict_image(img_path):
    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        feat = encoder(x)
        dists = torch.stack([F.pairwise_distance(feat, proto.unsqueeze(0)) for proto in prototypes.values()])
        pred_idx = torch.argmin(dists).item()
        pred_class = class_names[pred_idx]
        conf = torch.softmax(-dists, dim=0)[pred_idx].item()
    return pred_class, conf

# ====== Predict on data/check ======
print("\n--- FSL Predictions ---")
for filename in os.listdir(DATA_CHECK):
    if filename.lower().endswith((".png", ".jpg", ".jpeg")):
        path = os.path.join(DATA_CHECK, filename)
        pred, conf = predict_image(path)
        print(f"{filename}: {pred} (confidence: {conf:.3f})")


KeyboardInterrupt: 